# 🔄 A New Idea: The Omega Correction
### EPS Research High-School Exploration Track — Ages 12-14

Scientists at EPS Research found something interesting:
there's a pattern in how galaxy rotation curves behave.

They discovered a simple **correction** that can be calculated
from just two measurements — the innermost and outermost points
of a rotation curve.

This correction is called **omega (ω)** — the Greek letter that looks like a lowercase w.

Let's see it in action on our galaxy DDO161!

In [ ]:
# ── Colab setup: canonical FAIR² corpus paths ─────────────
import os, sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    import urllib.request
    CORPORA = {
        'rotation_curve_corpus_v7.json': 'https://zenodo.org/records/19563417/files/rotation_curve_corpus_v7.json',
        'high_z_kinematic_corpus_Z1.json': 'https://zenodo.org/records/21834678/files/high_z_kinematic_corpus_Z1.json',
        'dwarf_irregular_corpus_v1.json': 'https://zenodo.org/records/20320362/files/dwarf_irregular_corpus_v1.json',
    }
    for filename, url in CORPORA.items():
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            urllib.request.urlretrieve(url, filename)
            print(f"  ✓ {filename}")
        else:
            print(f"  Already present: {filename}")

    HI_PATH = 'rotation_curve_corpus_v7.json'
    Z1_PATH = 'high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = 'dwarf_irregular_corpus_v1.json'
    print("Ready.")
else:
    HI_PATH = '../hi/rotation_curve_corpus_v7.json'
    Z1_PATH = '../highz/high_z_kinematic_corpus_Z1.json'
    DWARF_PATH = '../dwarfs/dwarf_irregular_corpus_v1.json'
    print("Running locally — using canonical repository corpus paths.")


In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

with open(HI_PATH) as f:
    corpus = json.load(f)

galaxy = next(g for g in corpus['galaxies'] if g['galaxy'] == 'DDO161')
data   = galaxy['data']

R    = np.array([p['Rad']  for p in data])
Vobs = np.array([p['Vobs'] for p in data])
errV = np.array([p['errV'] for p in data])

# The omega correction uses just two boundary points!
R1, V1 = R[0],  Vobs[0]   # innermost point
R2, V2 = R[-1], Vobs[-1]  # outermost point

# Calculate omega (the correction)
outer_term    = (V2 / R2)
inner_term    = (V1 / R1) * ((R1 / R2) ** 1.5)
omega_kms_kpc = outer_term - inner_term          # Flynn & Cannaliato 2025 Eq.6  [km/s/kpc]
omega_rad_gyr = omega_kms_kpc * 1.0227           # 1 km/s/kpc = 1.0227 rad/Gyr
omega = omega_rad_gyr  # reporting/storage only — use omega_kms_kpc for velocity arithmetic

print(f"Innermost point: R = {R1:.2f} kpc,  V = {V1:.1f} km/s")
print(f"Outermost point: R = {R2:.2f} kpc,  V = {V2:.1f} km/s")
print()
print(f"Omega (ω) = {omega:.3f} rad/Gyr")
print()
print("Now we apply the correction to get the adjusted velocity:")
V_adj = Vobs - R * omega_kms_kpc
print("V_adjusted = V_observed - R × ω")

In [ ]:
# Compare the original curve with the corrected curve
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(R, Vobs, yerr=errV, fmt='o', color='#3498db',
            capsize=3, markersize=5, label='🔵 Original observed speed', zorder=5)
ax.plot(R, V_adj, '^-', color='#2ecc71', linewidth=2, markersize=6,
        label=f'🟢 Omega-corrected speed (ω = {omega:.2f})')
ax.set_xlabel('Distance from center (kpc)', fontsize=12)
ax.set_ylabel('Speed (km/s)', fontsize=12)
ax.set_title('The Omega Correction — DDO161\n'
             'EPS Research Flynn & Cannaliato (2025)', fontsize=11)
ax.legend(fontsize=9)
ax.text(0.02, 0.08,
        'The green line brings the\nobserved speed closer to\nthe baryonic prediction!',
        transform=ax.transAxes, va='bottom', fontsize=8,
        bbox=dict(boxstyle='round', fc='lightyellow', alpha=0.9))
plt.tight_layout()
plt.savefig('hs_a_09_omega_correction.png', dpi=150, bbox_inches='tight')
plt.show()

## What did the omega correction do?

The green line shows the **corrected** rotation curve.
It's closer to what we'd expect from just the visible matter!

This is the EPS Research discovery: a simple two-point correction
that works across many different galaxies.

Scientists published this in a journal article in 2025:
**Flynn & Cannaliato (2025)** in *Frontiers in Astronomy and Space Sciences*.

In the final notebook, we'll look at what this means for the dark matter mystery! 🌑